In [1]:
%load_ext autoreload
%autoreload 2
import os
if not hasattr(__builtins__, '_cwd_set'):
    os.chdir('..')
    __builtins__._cwd_set = True

In [2]:
from pathlib import Path
from pprint import pprint

import httpx

In [3]:
from util import util
API_URL = 'http://127.0.0.1:8000/api/chat/response'

In [7]:
RAG_CHAT = 3
rag_response = httpx.post(
    API_URL,
    json={
        'text': 'What support is available for caring for my child at home?',
        'chat_type': RAG_CHAT,
        'metadata': {
            'search_indexes': ['find_relevant_laws', 'find_similar_cases'],
            'generate_followups': True,
        },
    },
    timeout=60,
)
rag_response.raise_for_status()
rag_result = rag_response.json()
print(rag_result['text'])
pprint(rag_result['follow_up_questions'])
pprint(rag_result['evidence'])

Depending on your child’s needs and circumstances, your municipality may provide:

- **Relief care in your home** if you care for a child or young person with reduced physical or mental functioning. Respite care outside the home may also be available. [guidelines.pdf#page=59]
- **Help with necessary additional expenses**, including employing a helper for relief care in your own home where other support or respite arrangements are insufficient. [guidelines.pdf#page=37]
- **Practical help with household tasks**, allowing you to focus on your child. This can supplement compensation for lost earnings, particularly where the child needs constant supervision. [guidelines.pdf#page=46]
- **Compensation for lost earnings** where it is most appropriate for a parent to care for the child at home. [case-110.json#page=1]
- **Approved home training** for a child with a significant and permanent disability, either wholly or partly at home. [guidelines.pdf#page=14] If necessary for approved training, 

In [1]:

DCR_CONTROLLER_CHAT = 2
query = "What support can I get to take care of my child at home?"
search_response = httpx.post(
    API_URL,
    json={'text': query, 'chat_type': DCR_CONTROLLER_CHAT},
    timeout=60,
)
search_response.raise_for_status()
search_result = search_response.json()
print(search_result['text'])
pprint(search_result['graphs'])

NameError: name 'httpx' is not defined

In [ ]:
selected_source = search_result['graphs'][2]['source']
selected_source

In [ ]:

request_body = {
    'text': selected_source,
    'session_id': search_result['session_id'],
    'dcr_role': 'Citizen'
}
selection_response = httpx.post(API_URL, json=request_body, timeout=60)
selection_response.raise_for_status()
execution_result = selection_response.json()

In [ ]:
session_id = execution_result['session_id']
dcr_graph = util.import_xml(execution_result['graph_xml'])
backend_marking = util.marking(dcr_graph)

In [ ]:
HISTORY_URL = 'http://127.0.0.1:8000/api/chat/history'
hist_response = httpx.post(HISTORY_URL, json={"session_id":session_id}, timeout=30)
hist_response.json()

In [ ]:
print(execution_result['text'])

In [ ]:
answer = "No"
request_body = {
    'text': answer,
    'session_id': session_id,
    'act_id': execution_result['act_id'],
    'dcr_role': 'Citizen'
}
timeout = httpx.Timeout(connect=10, read=60*4, write=30, pool=10)

with httpx.Client(timeout=timeout) as client:
    response = client.post(API_URL, json=request_body)
    response.raise_for_status()
    execution_result = response.json()

## System/LLM interaction boundaries with DCR Graphs

How would dcr graphs with multiple data variables look like?

In [ ]:
from pm4py.objects.dcr.ocdcr.semantics import DcrSemantics
from pm4py.objects.dcr.ocdcr import obj
from tools.summarize_case import SummarizeCaseHistory
from tools.find_similar_cases import FindSimilarCases
from tools.find_relevant_laws import FindRelevantLaws

dcr_semantics = DcrSemantics()
computation = [("source","tool")]

ed1 = obj.DcrEventData(name="age", data_type=int)
act1 = obj.DcrActivity("Event_act1",label="Age",role="Citizen", 
                       takesInput=True, 
                       eventData=ed1,
                       priority=1)

ed2 = obj.DcrEventData(name="threshold", data_type=bool)
act2 = obj.DcrActivity("Event_act2",label="Age Threshold",role="Robot", 
                       takesInput=True, 
                       eventData=ed2,
                       priority=2)
ed21 = obj.DcrEventData(name="relevant_laws", data_type=str)
act21 = obj.DcrActivity("Event_act21",label="Relevant laws",role="Robot", 
                    #    eventData=ed21,
                       priority=3, 
                       computation = [("source","tool")])
ed22 = obj.DcrEventData(name="relevant_cases", data_type=str)
act22 = obj.DcrActivity("Event_act22",label="Relevant cases",role="Robot", 
                    #    eventData=ed21,
                       priority=4, 
                       computation = [("source","tool")])

ed3 = obj.DcrEventData(name="summary", data_type=str)
act3 = obj.DcrActivity("Event_act3",label="Summarize case",role="Robot",
                    #    eventData=ed3,
                       priority=5, 
                       computation = [("source","tool","graph","executions")])
relations = {
    obj.DcrSetValue(act1, act2, [("source", "data"), ">=", 18]),
    obj.DcrConstraint(obj.RelationType.C, act2, act21, guard=[("source", "data"), "==", True]),
    obj.DcrConstraint(obj.RelationType.C, act2, act22, guard=[("source", "data"), "==", True]),
    obj.DcrConstraint(obj.RelationType.C, act1, act2),
    obj.DcrConstraint(obj.RelationType.C, act2, act3),
    obj.DcrConstraint(obj.RelationType.C, act22, act3),
    obj.DcrConstraint(obj.RelationType.C, act21, act3),
}
act21.description = "Find relevant laws about an underage child that had access to alcohol."
act22.description = "Find relevant cases where an underage child was drunk." 
act3.description = "Make a brief summary of this case!"
act21.tool_call = FindRelevantLaws().answer
act22.tool_call = FindSimilarCases().answer
act3.tool_call = SummarizeCaseHistory().get_summary

activities = {act1, act2, act21, act22, act3}
graph = obj.DcrGraph("testGraph", elements=activities,relations=relations)

In [ ]:
dcr_semantics.executeActivity(obj.DcrExecution("Event_act1", 18), graph)

In [ ]:
dcr_semantics.executeActivity(obj.DcrExecution("Event_act2"), graph)

In [ ]:
dcr_semantics.executeActivity(obj.DcrExecution("Event_act21"), graph)

In [ ]:

dcr_semantics.executeActivity(obj.DcrExecution("Event_act22"), graph)


In [ ]:

dcr_semantics.executeActivity(obj.DcrExecution("Event_act3"), graph)

In [ ]:
for e in graph.executions:
    act = graph.getActivity(e.activityID)
    print("----------------------")
    print(act.label, "\n",act.description ,"\n",act.computation,act.tool_call,"\n",act.data)

In [ ]:
t = [e.activityID for e in graph.executions]

In [ ]:
h = [graph.getActivity(x).data for x in [e.activityID for e in graph.executions]]

In [ ]:
from pm4py.objects.dcr.exporter import exporter as dcr_exporter
dcr_exporter.apply(graph,'/home/vco/Projects2026/DcrController/backend/data/models/ToolCallTest.xml'
,dcr_exporter.Variants.DCR_JS_PORTAL)

In [ ]:
from util.csvparser import CsvParser
from util.fileprocessor import FileProcessor
from util.jsonparser import JsonParser
from util.pdfparser import LocalPdfParser
from util.textparser import TextParser
from util.textsplitter import SentenceTextSplitter, SimpleTextSplitter, XmlSplitter

csv_max_chars_per_page = 1000
sentence_text_splitter = SentenceTextSplitter()
file_processors = {
    ".json": FileProcessor(JsonParser(), SimpleTextSplitter()),
    ".xml": FileProcessor(TextParser(), XmlSplitter()),
    ".md": FileProcessor(TextParser(), sentence_text_splitter),
    ".txt": FileProcessor(TextParser(), sentence_text_splitter),
    ".csv": FileProcessor(CsvParser(max_chars_per_page=csv_max_chars_per_page), sentence_text_splitter),
    ".pdf": FileProcessor(LocalPdfParser(), sentence_text_splitter),
}

In [ ]:
fp = FileProcessor(LocalPdfParser(), sentence_text_splitter)

In [ ]:
fp.parser.parse()

In [ ]:
from tools.find_similar_cases import FindSimilarCases

finder = FindSimilarCases()

results = finder.find("child disability expenses", top_k=5)
clusters = finder.cluster("child disability expenses", top_k_per_outcome=5)

print(clusters.positive)
print(clusters.negative)
print(clusters.unknown)
for result in results:
    print(result.score, result.source, result.page_number)#, result.text)

In [ ]:
from tools.find_relevant_laws import FindRelevantLaws

results = FindRelevantLaws().find("requirements for compensation", top_k=5)

for result in results:
    print(result.score, result.source, result.page_number)#, result.text)

In [ ]:
delete_response = httpx.request(
    'DELETE',
    'http://127.0.0.1:8000/api/chat/session',
    json={'session_id': execution_result['session_id']},
)
delete_response.raise_for_status()
print('Session removed from backend memory.')

In [ ]:
from sentence_transformers import SentenceTransformer

# Download from the 🤗 Hub
model = SentenceTransformer("google/embeddinggemma-300m")

# 2. Save to a local directory
model.save_pretrained("models/local_gemma_embedding")
print("Model successfully saved locally!")

In [ ]:
model = SentenceTransformer("models/local_gemma_embedding")
# Run inference with queries and documents
query = "Which planet is known as the Red Planet?"
documents = [
    "Venus is often called Earth's twin because of its similar size and proximity.",
    "Mars, known for its reddish appearance, is often referred to as the Red Planet.",
    "Jupiter, the largest planet in our solar system, has a prominent red spot.",
    "Saturn, famous for its rings, is sometimes mistaken for the Red Planet."
]
query_embeddings = model.encode_query(query)
document_embeddings = model.encode_document(documents)
print(query_embeddings.shape, document_embeddings.shape)
# (768,) (4, 768)

# Compute similarities to determine a ranking
similarities = model.similarity(query_embeddings, document_embeddings)
print(similarities)
# tensor([[0.3011, 0.6359, 0.4930, 0.4889]])
